# 🏥 YumiCare — Genuine Fetal Ultrasound GAN + Segmentation Training
### Real GPU Training | No Fake Fallback | Class-Balanced Pipeline

**Project:** YumiCare Credibility Benchmark  
**Hardware Target:** Google Colab GPU (T4 / A100 recommended)  

---

### 📊 Dataset Class Imbalance (root cause of poor past results)
| Class | Pixel % | Status |
|---|---|---|
| Background | 70.97% | Dominant |
| Brain (Red) | 28.93% | OK |
| **CSP (Green)** | **0.10%** | ⚠️ Severely underrepresented |
| **LV (Blue)** | **0.004%** | ⚠️ Almost absent |

### ✅ Fixes Applied
1. **cWGAN-GP** with upgraded Generator (base_dim=96, CBAM attention) + Spectral Norm Critic
2. **WeightedRandomSampler** — CSP/LV-rich images pulled 20x more frequently  
3. **Minority mask synthesis boost** — 40% of synthetic masks have enlarged CSP/LV
4. **Combined Loss**: Weighted CrossEntropy + Soft Dice — both handle tiny minority classes
5. **Full epoch training** — no 15/30 batch caps from previous code
6. **CosineAnnealingLR** scheduler for both GAN and segmentation training

---

### 📋 Pipeline
1. Setup + Dataset Analysis
2. Train cWGAN-GP (50 epochs) → generate 500 synthetic pairs
3. Train 5 segmentation models × 3 regimes (15 epochs each)
4. Evaluate + export genuine results
5. Download report


In [ ]:
# ============================================================
# CELL 1: Install dependencies & verify GPU
# ============================================================
!pip install -q openpyxl pandas scipy pillow matplotlib tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import pandas as pd
import random, os, time, collections
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — training will be very slow on CPU!')

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive + Clone Public Repo
# ============================================================
from google.colab import drive
import os
import sys
from pathlib import Path

drive.mount('/content/drive')

REPO_URL = 'https://github.com/itzzSPcoder/YumiCare.git'
WORK_DIR = Path('/content/YumiCare')

if not WORK_DIR.exists():
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull

sys.path.insert(0, str(WORK_DIR / 'scripts'))
os.chdir(WORK_DIR / 'scripts')
print(f'✅ Working dir: {os.getcwd()}')


In [ ]:
# ============================================================
# CELL 3: Dataset Analysis — Verify Imbalance
# ============================================================
DATASET_DIR = WORK_DIR / 'Dataset' / '8265464'

# Count images per scan type
view_counts = {}
for subgroup in DATASET_DIR.glob('*'):
    if not subgroup.is_dir(): continue
    seg = next(subgroup.glob('*-Segmentation'), None)
    if not seg: continue
    mask_dir = seg / 'SegmentationClass'
    if mask_dir.exists():
        view_counts[subgroup.name] = len(list(mask_dir.glob('*.png')))

print('📊 View-level image counts:')
for k, v in sorted(view_counts.items(), key=lambda x: -x[1]):
    print(f'   {k}: {v} images')

# Pixel-level class distribution (sample 100 masks)
pixel_counts = collections.Counter()
sampled = 0
for subgroup in DATASET_DIR.glob('*'):
    seg = next(subgroup.glob('*-Segmentation'), None)
    if not seg: continue
    mask_dir = seg / 'SegmentationClass'
    if not mask_dir.exists(): continue
    for mp in list(mask_dir.glob('*.png'))[:25]:
        arr = np.array(Image.open(mp).convert('RGB'))
        pixel_counts['Background'] += int((arr.sum(-1) == 0).sum())
        pixel_counts['Brain (Red)'] += int(np.all(arr == [255,0,0], axis=-1).sum())
        pixel_counts['CSP (Green)'] += int(np.all(arr == [0,255,0], axis=-1).sum())
        pixel_counts['LV (Blue)']   += int(np.all(arr == [0,0,255], axis=-1).sum())
        sampled += 1

total_px = sum(pixel_counts.values())
print(f'\n📊 Pixel-level class distribution ({sampled} masks sampled):')
for cls, count in pixel_counts.items():
    print(f'   {cls}: {count:,} px ({100*count/total_px:.3f}%)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(view_counts.keys(), view_counts.values(), color=['#2196F3','#4CAF50','#FF9800','#E91E63'])
axes[0].set_title('View-Level Image Count', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylabel('# Images')

colors = ['#607D8B', '#F44336', '#4CAF50', '#2196F3']
wedges, texts, autotexts = axes[1].pie(
    pixel_counts.values(), labels=pixel_counts.keys(),
    colors=colors, autopct='%1.3f%%', startangle=90
)
axes[1].set_title('Pixel-Level Class Distribution\n(Note: CSP + LV < 0.15%!)', fontweight='bold')
plt.tight_layout()
    (WORK_DIR / 'runs').mkdir(parents=True, exist_ok=True)
plt.savefig(WORK_DIR / 'runs' / 'dataset_imbalance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset analysis complete')

In [ ]:
# ============================================================
# CELL 4: Load Genuine Training Modules (from scripts/)
# ============================================================
from fetal_gan_architecture import (
    UltrasoundMaskConditionedGenerator,
    UltrasoundPatchCritic,
    compute_gradient_penalty,
    compute_class_weights,
    DiceLoss,
)
from train_fetal_gan import (
    FetalUltrasoundGANDataset,
    synthesize_synthetic_anatomical_mask,
    train_fetal_gan,
    generate_synthetic_dataset,
    save_gan_synthesis_preview,
)
from run_segmentation_benchmark import (
    run_benchmark,
    create_premium_excel_report,
    save_visualization_plots,
    DATASET_PIXEL_COUNTS,
)
print('✅ All modules loaded')

In [ ]:
# ============================================================
# CELL 5: Hyperparameter Configuration
# ============================================================

# ── GAN Training ──────────────────────────────────────────
GAN_EPOCHS      = 50       # Colab T4: ~45 min | A100: ~15 min
GAN_BATCH_SIZE  = 8
GAN_LR          = 1e-4
GAN_LAMBDA_GP   = 10.0     # WGAN-GP penalty weight
GAN_LAMBDA_L1   = 50.0     # Pixel reconstruction term
GAN_CRITIC_ITER = 5        # Critic steps per generator step
GAN_GENERATE    = 500      # Synthetic pairs to produce
GAN_MINORITY_R  = 0.40     # 40% generated with enlarged CSP/LV

# ── Segmentation Training ─────────────────────────────────
SEG_EPOCHS      = 15       # Per model per regime
SEG_BATCH_SIZE  = 8
SEG_LR          = 1e-4
IMAGE_SIZE      = 128

# ── Paths ────────────────────────────────────────────────
RUNS_DIR = WORK_DIR / 'runs'
SYN_DIR  = WORK_DIR / 'Dataset' / 'synthetic_gan'
RUNS_DIR.mkdir(exist_ok=True)

print('📋 Configuration:')
print(f'   GAN: {GAN_EPOCHS} epochs | batch={GAN_BATCH_SIZE} | generate={GAN_GENERATE} pairs')
print(f'   Seg: {SEG_EPOCHS} epochs × 5 models × 3 regimes = {SEG_EPOCHS*5*3} total training runs')
print(f'   Output: {RUNS_DIR}')

In [ ]:
# ============================================================
# CELL 6: Train cWGAN-GP — Genuine GAN Training
# ============================================================
# This is the REAL GAN training — no shortcuts, no fake weights.
# Generator: base_dim=96, 6 res blocks, CBAM attention
# Critic: Spectral Norm + Multi-scale PatchGAN
# Sampler: WeightedRandomSampler (CSP/LV oversampled 20x)

print('🚀 Starting genuine cWGAN-GP training...')
print(f'   WeightedRandomSampler: CSP/LV-rich samples oversampled up to 20x')
print(f'   Gradient Penalty lambda={GAN_LAMBDA_GP}')
print()

gan_start = time.time()

generator, critic = train_fetal_gan(
    dataset_dir   = DATASET_DIR,
    output_dir    = RUNS_DIR,
    epochs        = GAN_EPOCHS,
    batch_size    = GAN_BATCH_SIZE,
    lr            = GAN_LR,
    lambda_gp     = GAN_LAMBDA_GP,
    lambda_l1     = GAN_LAMBDA_L1,
    critic_iters  = GAN_CRITIC_ITER,
    dry_run       = False,
)

gan_time = time.time() - gan_start
print(f'\n✅ GAN training complete in {gan_time/60:.1f} minutes')

In [ ]:
# ============================================================
# CELL 7: Generate Synthetic Dataset (class-balanced)
# ============================================================
print(f'🎨 Generating {GAN_GENERATE} synthetic ultrasound pairs...')
print(f'   Minority boost ratio: {GAN_MINORITY_R*100:.0f}% of samples will have enlarged CSP/LV')

gen_stats = generate_synthetic_dataset(
    generator      = generator,
    output_dir     = SYN_DIR,
    num_samples    = GAN_GENERATE,
    image_size     = IMAGE_SIZE,
    minority_ratio = GAN_MINORITY_R,
)

print(f'\n📊 Synthetic generation stats:')
for k, v in gen_stats.items():
    print(f'   {k}: {v}')

# Save preview grid
save_gan_synthesis_preview(
    generator    = generator,
    output_path  = RUNS_DIR / 'gan_synthesis_preview.png',
    num_samples  = 6,
    image_size   = IMAGE_SIZE,
)

# Display preview
from IPython.display import Image as IPImage
IPImage(str(RUNS_DIR / 'gan_synthesis_preview.png'), width=700)

In [ ]:
# ============================================================
# CELL 8: Verify Synthetic Dataset — Post-generation analysis
# ============================================================
syn_img_dir  = SYN_DIR / 'images'
syn_mask_dir = SYN_DIR / 'masks'

syn_pixel_counts = collections.Counter()
syn_masks = list(syn_mask_dir.glob('*.png'))[:100]

for mp in syn_masks:
    arr = np.array(Image.open(mp).convert('RGB'))
    syn_pixel_counts['Background'] += int((arr.sum(-1) == 0).sum())
    syn_pixel_counts['Brain (Red)'] += int(np.all(arr == [255,0,0], axis=-1).sum())
    syn_pixel_counts['CSP (Green)'] += int(np.all(arr == [0,255,0], axis=-1).sum())
    syn_pixel_counts['LV (Blue)']   += int(np.all(arr == [0,0,255], axis=-1).sum())

total_syn = sum(syn_pixel_counts.values())
print('📊 Synthetic dataset pixel distribution (first 100 masks):')
for cls, count in syn_pixel_counts.items():
    pct = 100*count/total_syn
    print(f'   {cls}: {count:,} px ({pct:.2f}%)')

print('\n📊 Comparison: Real vs Synthetic class distribution')
total_real = sum(DATASET_PIXEL_COUNTS.values())
class_map = {0: 'Background', 1: 'Brain (Red)', 2: 'CSP (Green)', 3: 'LV (Blue)'}
for i, name in class_map.items():
    real_pct = 100 * DATASET_PIXEL_COUNTS[i] / total_real
    syn_pct  = 100 * syn_pixel_counts.get(name, 0) / (total_syn + 1e-8)
    arrow = '⬆️' if syn_pct > real_pct else '='  
    print(f'   {name}: Real={real_pct:.3f}% → Synthetic={syn_pct:.3f}% {arrow}')

In [ ]:
# ============================================================
# CELL 9: Genuine Segmentation Benchmark
# ============================================================
# Trains 5 models × 3 regimes with:
#   - WeightedCrossEntropy + SoftDice combined loss
#   - WeightedRandomSampler (oversamples CSP/LV images)
#   - Full epoch training (NO batch cap)
#   - CosineAnnealingLR scheduler

print('🚀 Starting genuine segmentation benchmark...')
print(f'   5 models × 3 regimes × {SEG_EPOCHS} epochs = {5*3*SEG_EPOCHS} total training epochs')
print(f'   Loss: 0.5 × WeightedCE + 0.5 × SoftDice')
print(f'   Class weights: BG~1.4, Brain~3.5, CSP~50.0, LV~80.0')
print()

seg_start = time.time()

results_df, per_class_df, ablation_df = run_benchmark(
    dataset_dir    = DATASET_DIR,
    output_dir     = RUNS_DIR,
    augmentation_mode = 'all',
    seg_epochs     = SEG_EPOCHS,
    seg_batch_size = SEG_BATCH_SIZE,
    seg_lr         = SEG_LR,
)

seg_time = time.time() - seg_start
print(f'\n✅ Segmentation benchmark complete in {seg_time/60:.1f} minutes')

In [ ]:
# ============================================================
# CELL 10: Display & Save Genuine Results
# ============================================================
print('📊 GENUINE BENCHMARK RESULTS (GAN-Augmented Synthesis regime):')
print('='*70)
display_cols = ['model', 'mean_iou', 'mean_dice', 'precision', 'recall', 'latency_ms']
print(results_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))

print('\n📊 ABLATION STUDY (all 3 regimes):')
print('='*70)
abl_cols = ['model', 'regime', 'mean_iou', 'mean_dice', 'delta_iou_vs_baseline']
print(ablation_df[abl_cols].to_string(index=False, float_format='{:.4f}'.format))

# Save genuine CSV (clearly named)
genuine_csv = RUNS_DIR / 'genuine_benchmark_results.csv'
results_df.to_csv(genuine_csv, index=False)
ablation_df.to_csv(RUNS_DIR / 'genuine_ablation_results.csv', index=False)
per_class_df.to_csv(RUNS_DIR / 'genuine_per_class_results.csv', index=False)
print(f'\n✅ Results saved to: {genuine_csv}')

# Best model
best = results_df.loc[results_df['mean_iou'].idxmax()]
print(f'\n🏆 Best Model: {best["model"]}')
print(f'   Mean IoU:  {best["mean_iou"]:.4f}')
print(f'   Mean Dice: {best["mean_dice"]:.4f}')

In [ ]:
# ============================================================
# CELL 11: Visualize Results
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Genuine Training Results — YumiCare Fetal Segmentation Benchmark',
             fontsize=14, fontweight='bold')

# Plot 1: IoU + Dice comparison
x = np.arange(len(results_df))
w = 0.35
axes[0].bar(x - w/2, results_df['mean_iou']*100, w, label='Mean IoU (%)', color='#1F4E79')
axes[0].bar(x + w/2, results_df['mean_dice']*100, w, label='Mean Dice (%)', color='#70AD47')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['model'], rotation=20, ha='right')
axes[0].set_ylabel('Score (%)')
axes[0].set_title('IoU & Dice (GAN-Augmented Regime)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

# Plot 2: Ablation — IoU across regimes
models  = ablation_df['model'].unique()
regimes = ablation_df['regime'].unique()
x2 = np.arange(len(models))
w2 = 0.25
colors2 = ['#BDD7EE', '#2E75B6', '#1F4E79']
for ri, (regime, color) in enumerate(zip(regimes, colors2)):
    vals = [ablation_df[(ablation_df['model']==m) & (ablation_df['regime']==regime)]['mean_iou'].values[0]
            for m in models]
    axes[1].bar(x2 + (ri-1)*w2, np.array(vals)*100, w2, label=regime, color=color)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(models, rotation=20, ha='right')
axes[1].set_ylabel('Mean IoU (%)')
axes[1].set_title('Ablation: IoU Across Training Regimes')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.4)

# Plot 3: Latency vs IoU scatter
for _, row in results_df.iterrows():
    axes[2].scatter(row['latency_ms'], row['mean_iou']*100, s=200, alpha=0.85)
    axes[2].annotate(row['model'], (row['latency_ms']+0.5, row['mean_iou']*100+0.2), fontsize=8)
axes[2].set_xlabel('Inference Latency (ms)')
axes[2].set_ylabel('Mean IoU (%)')
axes[2].set_title('Clinical Pareto: IoU vs Latency')
axes[2].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(RUNS_DIR / 'genuine_results_visualization.png', dpi=200, bbox_inches='tight')
plt.show()
print('✅ Plots saved')

In [ ]:
# ============================================================
# CELL 12: Export Premium Excel Report
# ============================================================
excel_path = RUNS_DIR / 'genuine_fetal_segmentation_report.xlsx'
create_premium_excel_report(results_df, per_class_df, ablation_df, excel_path)
print(f'✅ Excel report saved: {excel_path}')

In [ ]:
# ============================================================
# CELL 13: Download Results to Local Machine
# ============================================================
from google.colab import files
import zipfile

# Zip all key outputs
zip_path = RUNS_DIR / 'genuine_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in [
        'genuine_benchmark_results.csv',
        'genuine_ablation_results.csv',
        'genuine_per_class_results.csv',
        'genuine_fetal_segmentation_report.xlsx',
        'genuine_results_visualization.png',
        'gan_synthesis_preview.png',
        'dataset_imbalance_analysis.png',
    ]:
        fp = RUNS_DIR / f
        if fp.exists():
            zf.write(fp, f)
            print(f'  Added: {f}')

print(f'\n📦 Downloading results zip...')
files.download(str(zip_path))

In [ ]:
# ============================================================
# CELL 14: Commit Results back to GitHub (optional)
# ============================================================
# Uncomment and set your GitHub token to auto-commit results

# GITHUB_TOKEN = 'ghp_YOUR_TOKEN_HERE'
# !git -C {WORK_DIR} config user.email 'colab@yumicare.ai'
# !git -C {WORK_DIR} config user.name 'YumiCare Colab'

# # Copy results back to repo
# import shutil
# for f in RUNS_DIR.glob('genuine_*.csv'):
#     shutil.copy(f, WORK_DIR / 'runs' / f.name)
# for f in RUNS_DIR.glob('genuine_*.xlsx'):
#     shutil.copy(f, WORK_DIR / 'runs' / f.name)

# !git -C {WORK_DIR} add runs/genuine_*.csv runs/genuine_*.xlsx runs/*.png
# !git -C {WORK_DIR} commit -m 'chore: add genuine GPU training results [no fake values]'
# !git -C {WORK_DIR} push https://{GITHUB_TOKEN}@github.com/itzzSPcoder/YumiCare.git main
# print('✅ Results committed to GitHub')

print('💡 Uncomment the cells above to auto-commit results to GitHub')